# Sagemaker linear estimator regressor test.

In [1]:
import pandas as pd
import boto3
import sagemaker
import os
import io
import numpy as np
import sagemaker.amazon.common as smac 

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml


In [2]:
sagemaker_session = sagemaker.Session()
role = sagemaker.get_execution_role()
bucket_name = "sagemaker-studio-533267358966-xpqdotx3pf"

In [3]:
X = np.arange(0, 30, 1)
X = X.reshape(-1, 1)
y = X + 0.2*np.random.rand(len(X), 1)
y = y.flatten()  # y needs to be 1D array.

**Storing numpy array in buffer**

In [4]:
buf = io.BytesIO()
smac.write_numpy_to_dense_tensor(buf, X, y)
buf.seek(0)

0

**Saving the buffered data to s3**

In [5]:
file_path = os.path.join('momory_folder', 'memory_data')
boto3.resource('s3').Bucket(bucket_name).Object(file_path).upload_fileobj(buf)
memory_path = f"s3://{bucket_name}/memory_folder/memory_data"  # can be used later during training.

Below is how to retrieve the data saved in s3.

In [6]:
buf2 = io.BytesIO()
boto3.resource('s3').Bucket(bucket_name).Object(file_path).download_fileobj(buf2)
buf2.seek(0)
records = smac.read_records(buf2)

In [7]:
print(records[0])

features {
  key: "values"
  value {
    int32_tensor {
      values: 0
    }
  }
}
label {
  key: "values"
  value {
    float64_tensor {
      values: 0.04606681425840602
    }
  }
}



In [8]:
X2 = []
y2 = []
for record in records:
    X2.append( record.features['values'].int32_tensor.values[0] )
    y2.append( record.label['values'].float64_tensor.values[0] )

**Saving the data to s3**

In [9]:
from sklearn.model_selection import train_test_split

train_features, test_features, train_labels, test_labels = train_test_split(X, y, test_size=0.2, random_state=42)
train_data = np.column_stack((train_labels, train_features)) # It needs to be label column first followed by features.
test_data = np.column_stack((test_labels, test_features))

train_file = 'train_data.csv'
test_file = 'test_data.csv'
pd.DataFrame(train_data).to_csv(train_file, header=False, index=False)
pd.DataFrame(test_data).to_csv(test_file, header=False, index=False)

In [10]:
sub_folder = 'data'
s3_train_data = sagemaker_session.upload_data(train_file, bucket=bucket_name, key_prefix=sub_folder)
s3_test_data = sagemaker_session.upload_data(test_file, bucket=bucket_name, key_prefix=sub_folder)

**Get the image**

In [11]:
from sagemaker import image_uris
container = image_uris.retrieve(region=boto3.Session().region_name, framework='linear-learner',version='1')

**Prepare for training and tuning**

Sagemaker Built-in algorithm.  https://sagemaker.readthedocs.io/en/stable/algorithms/index.html 

parameters are the here.  https://sagemaker.readthedocs.io/en/stable/algorithms/tabular/linear_learner.html

In [12]:
estimator = sagemaker.estimator.Estimator(
    container,
    role,
    instance_count=1,
    instance_type="ml.m5.large",
    output_path=f"s3://{bucket_name}/output", # Model file that will be saved.
    sagemaker_session=sagemaker_session,
    #use_spot_instances=True, 
    #max_wait=3600,  # max wait times for spot instance.
    )

In [13]:
estimator.set_hyperparameters(
    predictor_type="regressor",
    loss="squared_loss",
    normalize_data=True,
    mini_batch_size=5,
    )

In [14]:
from sagemaker.tuner import HyperparameterTuner, ContinuousParameter

# wd is the alpha in ridge loss function.  See the above doc for list of parameters.
hyperparameter_ranges = {"wd": ContinuousParameter(1e-6, 1.0)}

tuner = HyperparameterTuner(
    estimator=estimator,
    objective_metric_name="test:mse",  # if validation, choose validation:mse 
    objective_type='Minimize',
    hyperparameter_ranges=hyperparameter_ranges,
    max_jobs=1,
    max_parallel_jobs=1,
    )

**Set the input format to text/csv since the input data is saved in csv format.**

You don't need to do this if using buffered inputs.

In [16]:
train_input = sagemaker.inputs.TrainingInput(
    s3_train_data,
    content_type="text/csv",
    )
test_input = sagemaker.inputs.TrainingInput(
    s3_test_data,
    content_type="text/csv",
    )

**Train**

You can check the log in CloudWatch in case error happens.

In [ ]:
tuner.fit({
    'train': train_input,  # you may use memory data "memory_path" file in s3.
    'test': test_input  # you could change name to validation instead. 
    })

No finished training job found associated with this estimator. Please make sure this estimator is only used for building workflow config
No finished training job found associated with this estimator. Please make sure this estimator is only used for building workflow config


.......................................!


In [18]:
# After tuning completes, get the best model
best_training_job = tuner.best_training_job()
best_model = tuner.best_estimator()


2025-01-13 12:46:56 Starting - Preparing the instances for training
2025-01-13 12:46:56 Downloading - Downloading the training image
2025-01-13 12:46:56 Training - Training image download completed. Training in progress.
2025-01-13 12:46:56 Uploading - Uploading generated training model
2025-01-13 12:46:56 Completed - Resource released due to keep alive period expiry


In [19]:
from sagemaker.serializers import CSVSerializer
from sagemaker.deserializers import JSONDeserializer

predictor = best_model.deploy(
    initial_instance_count=1,
    instance_type='ml.t2.medium',
    serializer=CSVSerializer(),
    deserializer=JSONDeserializer(),
    #endpoint_name='linear_learner_ridge',
    )

------------------!

In [20]:
predictor.predict(test_features)

{'predictions': [{'score': 17.780048370361328},
  {'score': 14.578554153442383},
  {'score': 16.71288299560547},
  {'score': 15.112136840820312},
  {'score': 12.711015701293945},
  {'score': 12.97780704498291}]}